In [ ]:
from google.colab import files
uploaded = files.upload()

import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!apt-get -qq install fonts-nanum

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

sns.set_theme(
    style='whitegrid',
    font='NanumGothic'
)

In [ ]:
file_name = list(uploaded.keys())[0]

df = pd.read_excel(
    io.BytesIO(uploaded[file_name]),
    sheet_name='데이터',
    header=[0, 1, 2, 3]
)

# 다중 헤더의 공백 제거 및 문자열 통일
df.columns = pd.MultiIndex.from_tuples([
    tuple(str(value).strip() for value in column)
    for column in df.columns
])

df.head()

In [ ]:
def get_numeric_column(year, category, detail='소계', gender='계'):
    """다중 헤더에서 원하는 열을 숫자형으로 추출"""

    series = df[(str(year), category, detail, gender)].copy()

    # '-'는 자료 없음, '*'는 5명 미만 비공개
    series = series.replace({
        '-': np.nan,
        '*': np.nan
    })

    return pd.to_numeric(series, errors='coerce')


analysis = pd.DataFrame({
    '구군': df.iloc[:, 1].astype(str).str.strip()
})

# 연도별 총인구와 외국인 주민
for year in [2022, 2023, 2024]:
    analysis[f'총인구_{year}'] = get_numeric_column(
        year, '총인구 (명)'
    )

    analysis[f'외국인주민_{year}'] = get_numeric_column(
        year, '합계 (명)'
    )

# 2024년 외국인 유형
categories = [
    '외국인근로자',
    '결혼이민자',
    '유학생',
    '외국국적동포',
    '기타외국인'
]

for category in categories:
    analysis[category] = get_numeric_column(
        2024,
        '한국국적을 가지지 않은 자 (명)',
        category
    )

analysis['귀화자'] = get_numeric_column(
    2024,
    '한국국적을 취득한 자 (명)'
)

analysis['외국인주민자녀'] = get_numeric_column(
    2024,
    '외국인주민자녀(출생) (명)'
)

# 파생지표
analysis['외국인비율_2024'] = (
    analysis['외국인주민_2024']
    / analysis['총인구_2024']
    * 100
)

analysis['증가인원_2022_2024'] = (
    analysis['외국인주민_2024']
    - analysis['외국인주민_2022']
)

analysis['증가율_2022_2024'] = (
    analysis['외국인주민_2024']
    / analysis['외국인주민_2022']
    - 1
) * 100

# 대구 전체 소계와 구·군 분리
daegu_total = analysis[analysis['구군'] == '소계'].iloc[0]
district = analysis[analysis['구군'] != '소계'].copy()

district['대구외국인중_비중'] = (
    district['외국인주민_2024']
    / daegu_total['외국인주민_2024']
    * 100
)

district.round(2)

In [ ]:
summary = district[[
    '구군',
    '외국인주민_2024',
    '대구외국인중_비중',
    '외국인비율_2024',
    '증가인원_2022_2024',
    '증가율_2022_2024'
]].sort_values(
    '외국인주민_2024',
    ascending=False
)

summary.round(2)

In [ ]:
plot1 = district.sort_values(
    '외국인주민_2024',
    ascending=True
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    plot1['구군'],
    plot1['외국인주민_2024'],
    color='#2A9D8F'
)

plt.title(
    '2024년 대구 구·군별 외국인 주민 수',
    fontsize=16,
    fontweight='bold',
    pad=15
)
plt.xlabel('외국인 주민 수(명)')
plt.ylabel('')

for bar in bars:
    value = bar.get_width()
    plt.text(
        value + 200,
        bar.get_y() + bar.get_height()/2,
        f'{value:,.0f}명',
        va='center',
        fontsize=10
    )

plt.xlim(0, plot1['외국인주민_2024'].max() * 1.18)
plt.tight_layout()
plt.savefig('01_구군별_외국인주민수.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 대구 전체 합계만 추출
years = [2022, 2023, 2024]
foreign_population = [53684, 58944, 61554]

increase = foreign_population[-1] - foreign_population[0]
growth_rate = increase / foreign_population[0] * 100

fig, ax = plt.subplots(figsize=(10, 6))

# 영역과 선
ax.fill_between(
    years,
    foreign_population,
    50000,
    color='#50BFA5',
    alpha=0.18
)

ax.plot(
    years,
    foreign_population,
    color='#168F76',
    linewidth=4,
    marker='o',
    markersize=10
)

# 각 연도의 수치 표시
for year, value in zip(years, foreign_population):
    ax.text(
        year,
        value + 700,
        f'{value:,}명',
        ha='center',
        fontsize=13,
        fontweight='bold'
    )

# 증가율 강조
ax.annotate(
    f'2년간 +{increase:,}명\n({growth_rate:.1f}% 증가)',
    xy=(2024, 61554),
    xytext=(2023.25, 64500),
    fontsize=14,
    fontweight='bold',
    color='#D95D39',
    ha='center',
    arrowprops=dict(
        arrowstyle='->',
        color='#D95D39',
        linewidth=2
    )
)

ax.set_title(
    '대구 외국인 주민, 2024년 6만 명 돌파',
    fontsize=19,
    fontweight='bold',
    pad=22
)

ax.set_ylabel('외국인 주민 수(명)')
ax.set_xlabel('')
ax.set_xticks(years)

# 변화가 잘 보이도록 범위 조정
ax.set_ylim(50000, 67000)

ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', visible=False)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(
    '대구_외국인주민_증가추이.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
import io
import os
import re
import joblib
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("라이브러리 준비 완료")

In [ ]:
uploaded = files.upload()

In [ ]:
file_name = next(iter(uploaded))

df = pd.read_csv(
    io.BytesIO(uploaded[file_name]),
    encoding="utf-8-sig"
)

print("파일명:", file_name)
print("데이터 크기:", df.shape)
print("열 이름:", df.columns.tolist())

display(df.head())

In [ ]:
print("전체 문장 수:", len(df))
print("중복 문장 수:", df["text"].duplicated().sum())
print("결측 문장 수:", df["text"].isna().sum())

print("\n라벨별 개수")
print(df["label"].value_counts().sort_index())

In [ ]:
def clean_text(text):
    text = str(text)

    # 연속된 공백과 줄바꿈을 하나의 공백으로 변경
    text = re.sub(r"\s+", " ", text)

    # 문장 앞뒤 공백 제거
    text = text.strip()

    return text


df = df.dropna(subset=["text", "label"]).copy()

df["clean_text"] = df["text"].apply(clean_text)
df["label"] = df["label"].astype(int)

print("✅ 전처리 완료")

display(
    df[[
        "text",
        "clean_text",
        "label"
    ]].head()
)

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습 데이터:", len(X_train))
print("테스트 데이터:", len(X_test))

print("\n학습 데이터 라벨")
print(y_train.value_counts().sort_index())

print("\n테스트 데이터 라벨")
print(y_test.value_counts().sort_index())

In [ ]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 5),
            min_df=2,
            max_features=10000,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

print("✅ 모델 학습 완료")

In [ ]:
y_pred = model.predict(X_test)

y_probability = model.predict_proba(
    X_test
)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

display(metrics_df)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Normal", "Phishing"],
        digits=4,
        zero_division=0
    )
)

In [ ]:
plt.figure(figsize=(7, 4))

sns.barplot(
    data=metrics_df,
    x="Metric",
    y="Score",
    hue="Metric",
    palette=[
        "#9DB5A3",
        "#91B6D0",
        "#E59B7B",
        "#C4A9D9"
    ],
    legend=False
)

plt.title("Initial Test Performance")
plt.xlabel("Metric")
plt.ylabel("Score")
plt.ylim(0, 1.08)

for i, score in enumerate(metrics_df["Score"]):
    plt.text(
        i,
        score + 0.02,
        f"{score:.3f}",
        ha="center",
        fontweight="bold"
    )

plt.show()

In [ ]:
hard_test = pd.DataFrame({
    "text": [
        # 피싱 의심 10개
        "고객님의 기존 대출을 더 낮은 금리로 변경할 수 있습니다. 담당자 안내에 따라 기존 대출금을 지정된 계좌로 상환해 주세요.",

        "조금 전에 보낸 사진 확인했어? 휴대전화가 고장 나서 PC로 연락 중이야. 먼저 38만원만 대신 처리해 줘.",

        "본인 확인이 완료되지 않아 카드 사용이 제한될 예정입니다. 상담원이 보내드린 안내에 따라 확인 절차를 진행해 주세요.",

        "주문하신 물품의 배송 정보가 누락되었습니다. 아래 페이지에서 수령 정보를 다시 입력해 주세요.",

        "환급 가능한 세금이 조회되었습니다. 지급을 위해 사용하는 은행과 계좌 정보를 등록해 주세요.",

        "귀하 명의로 접수된 사건이 있어 자산 확인이 필요합니다. 담당 직원의 안내가 끝날 때까지 전화를 유지하세요.",

        "채용 절차가 완료되었습니다. 사원증과 장비 발급에 필요한 비용은 입사 후 돌려드릴 예정입니다.",

        "현재 보유하신 주식의 손실을 보상받을 수 있습니다. 안내받은 종목을 먼저 매수하시면 담당자가 처리합니다.",

        "공과금 납부 정보가 변경되었습니다. 이번 달부터 안내된 새 계좌를 이용해 주세요.",

        "해외 결제 취소를 접수하려면 상담용 프로그램을 실행한 뒤 화면에 표시되는 번호를 알려주세요.",

        # 정상 문자 10개
        "고객님이 요청한 계좌로 해외송금 500달러가 접수되었습니다. 신청한 거래가 아니라면 카드 뒷면 대표번호로 문의하세요.",

        "대출 원리금 420,000원이 자동이체될 예정입니다. 잔액을 확인해 주세요.",

        "본인인증이 완료되어 모바일뱅킹 이용 제한이 해제되었습니다.",

        "택배 배송 주소 변경이 완료되었습니다. 변경 내역은 주문한 쇼핑몰 앱에서 확인할 수 있습니다.",

        "건강보험료 환급금이 등록하신 계좌로 지급되었습니다.",

        "새 휴대전화에서 로그인이 감지되었습니다. 본인이 아니라면 직접 공식 앱을 실행하여 접속을 차단하세요.",

        "9월 전기요금 68,400원을 가상계좌로 납부할 수 있습니다. 계좌번호는 고객이 로그인한 공식 홈페이지에서만 제공합니다.",

        "온라인 금융교육 접속을 위한 안내 링크를 신청하신 이메일로 발송했습니다.",

        "카드 결제 취소가 완료되었으며 환급에는 영업일 기준 3일이 소요됩니다.",

        "비밀번호를 5회 잘못 입력하여 로그인이 제한되었습니다. 가까운 영업점에서 신분증 확인 후 해제할 수 있습니다."
    ],

    "label": [
        1, 1, 1, 1, 1,
        1, 1, 1, 1, 1,

        0, 0, 0, 0, 0,
        0, 0, 0, 0, 0
    ]
})

hard_test["clean_text"] = hard_test["text"].apply(
    clean_text
)

print("✅ 어려운 테스트셋 생성 완료")
print("데이터 크기:", hard_test.shape)
print(hard_test["label"].value_counts().sort_index())

display(hard_test.head())

In [ ]:
overlap = set(X_train).intersection(
    set(hard_test["clean_text"])
)

print(
    "학습 데이터와 완전히 동일한 문장 수:",
    len(overlap)
)

In [ ]:
hard_test["predicted"] = model.predict(
    hard_test["clean_text"]
)

hard_test["phishing_probability"] = model.predict_proba(
    hard_test["clean_text"]
)[:, 1]

hard_test["correct"] = (
    hard_test["label"]
    ==
    hard_test["predicted"]
)

display(
    hard_test[[
        "text",
        "label",
        "predicted",
        "phishing_probability",
        "correct"
    ]]
)

In [ ]:
hard_accuracy = accuracy_score(
    hard_test["label"],
    hard_test["predicted"]
)

hard_precision = precision_score(
    hard_test["label"],
    hard_test["predicted"],
    zero_division=0
)

hard_recall = recall_score(
    hard_test["label"],
    hard_test["predicted"],
    zero_division=0
)

hard_f1 = f1_score(
    hard_test["label"],
    hard_test["predicted"],
    zero_division=0
)

hard_metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    "Score": [
        hard_accuracy,
        hard_precision,
        hard_recall,
        hard_f1
    ]
})

display(hard_metrics_df)

print(
    classification_report(
        hard_test["label"],
        hard_test["predicted"],
        target_names=["Normal", "Phishing"],
        digits=4,
        zero_division=0
    )
)

In [ ]:
hard_cm = confusion_matrix(
    hard_test["label"],
    hard_test["predicted"]
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    hard_cm,
    annot=True,
    fmt="d",
    cmap="YlOrBr",
    xticklabels=["Normal", "Phishing"],
    yticklabels=["Normal", "Phishing"]
)

plt.title("Hard Test Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

In [ ]:
wrong_hard_test = hard_test[
    hard_test["correct"] == False
].copy()

print("틀린 문장 수:", len(wrong_hard_test))

display(
    wrong_hard_test[[
        "text",
        "label",
        "predicted",
        "phishing_probability"
    ]]
)

In [ ]:
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

feature_names = np.array(
    vectorizer.get_feature_names_out()
)

coefficients = classifier.coef_[0]

phishing_indices = np.argsort(
    coefficients
)[-20:][::-1]

normal_indices = np.argsort(
    coefficients
)[:20]

phishing_features = pd.DataFrame({
    "Feature": feature_names[phishing_indices],
    "Weight": coefficients[phishing_indices]
})

normal_features = pd.DataFrame({
    "Feature": feature_names[normal_indices],
    "Weight": coefficients[normal_indices]
})

print("피싱 판단에 영향을 준 표현")
display(phishing_features)

print("정상 판단에 영향을 준 표현")
display(normal_features)

In [ ]:
def predict_message(message):
    cleaned_message = clean_text(message)

    probability = float(
        model.predict_proba(
            [cleaned_message]
        )[0][1]
    )

    if probability >= 0.7:
        risk_level = "높음"
        prediction = "피싱 의심"

    elif probability >= 0.4:
        risk_level = "주의"
        prediction = "추가 확인 필요"

    else:
        risk_level = "낮음"
        prediction = "정상 가능성"

    return {
        "입력 문자": message,
        "판정": prediction,
        "피싱 확률": f"{probability * 100:.1f}%",
        "위험도": risk_level
    }

In [ ]:
predict_message(
    "미납요금이 있습니다. 오늘 안에 아래 계좌로 송금해 주세요."
)

In [ ]:
joblib.dump(
    model,
    "daeguide_phishing_model.pkl"
)

hard_metrics_df.to_csv(
    "hard_test_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

hard_test.to_csv(
    "hard_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ 파일 저장 완료")

In [ ]:
files.download(
    "daeguide_phishing_model.pkl"
)

In [ ]:
files.download(
    "hard_test_metrics.csv"
)

In [ ]:
files.download(
    "hard_test_predictions.csv"
)